# MIMICSplus — Mortality file preprocessing

Reads CLM h1 files (preprocessed by `preprocess_clm_input.bash`) and writes
per-year mortality `.nc` files in the format expected by MIMICSplus.

**Edit the CONFIGURATION cell before running.**

In [19]:
import os
import glob
import xarray as xr
import numpy as np

## Configuration

In [20]:
# ── Site settings ──────────────────────────────────────────────────────────
SITE = "Bygland"

# Directory containing preprocessed h1 files (output of preprocess_clm_input.bash)
# Files are expected to match the pattern: *.clm2.h1.{YEAR}.nc
H1_DIR = f"/home/elisacw/mimicsplus_input/{SITE}/clm_h1"

# Where to write per-year mortality files
MORT_DIR = f"/home/elisacw/mimicsplus_input/{SITE}/mortality"

# Year range to process (inclusive)
YEAR_START = 1850
YEAR_END   = 2025

# The first year of the historical run — used to extract profiles (CROOT_PROF, STEM_PROF)
# which are constant but only need to be written once.
PROFILE_YEAR = 1850

# Spinup years (used to build the spinup mortality file)
SPINUP_START = 1850
SPINUP_END   = 1869

os.makedirs(MORT_DIR, exist_ok=True)
print(f"Site:       {SITE}")
print(f"Input dir:  {H1_DIR}")
print(f"Output dir: {MORT_DIR}")

Site:       Bygland
Input dir:  /home/elisacw/mimicsplus_input/Bygland/clm_h1
Output dir: /home/elisacw/mimicsplus_input/Bygland/mortality


## Variable definitions

In [ ]:
# Variables to split between metabolic and structural litter
SPLIT_C = ["M_LEAFC_TO_LITTER",  "M_FROOTC_TO_LITTER"]
SPLIT_N = ["M_LEAFN_TO_LITTER",  "M_FROOTN_TO_LITTER"]

# Variables that go to metabolic litter only, grouped by the root profile they follow.
# Each key becomes a field in the output dataset.
MET_C = {
    "met_leaf_prof_mortC":  ["M_LEAFC_STORAGE_TO_LITTER",
                              "M_LEAFC_XFER_TO_LITTER",
                              "M_GRESP_STORAGE_TO_LITTER",
                              "M_GRESP_XFER_TO_LITTER"],
    "met_froot_prof_mortC": ["M_FROOTC_STORAGE_TO_LITTER",
                              "M_FROOTC_XFER_TO_LITTER"],
    "met_croot_prof_mortC": ["M_LIVECROOTC_XFER_TO_LITTER",
                              "M_DEADCROOTC_XFER_TO_LITTER",
                              "M_LIVECROOTC_STORAGE_TO_LITTER",
                              "M_DEADCROOTC_STORAGE_TO_LITTER"],
    "met_stem_prof_mortC":  ["M_LIVESTEMC_STORAGE_TO_LITTER",
                              "M_LIVESTEMC_XFER_TO_LITTER",
                              "M_DEADSTEMC_STORAGE_TO_LITTER",
                              "M_DEADSTEMC_XFER_TO_LITTER"],
}

MET_N = {
    "met_leaf_prof_mortN":  ["M_LEAFN_STORAGE_TO_LITTER",
                              "M_LEAFN_XFER_TO_LITTER",
                              "M_RETRANSN_TO_LITTER"],
    "met_froot_prof_mortN": ["M_FROOTN_STORAGE_TO_LITTER",
                              "M_FROOTN_XFER_TO_LITTER"],
    "met_croot_prof_mortN": ["M_LIVECROOTN_STORAGE_TO_LITTER",
                              "M_DEADCROOTN_STORAGE_TO_LITTER",
                              "M_LIVECROOTN_XFER_TO_LITTER",
                              "M_DEADCROOTN_XFER_TO_LITTER"],
    "met_stem_prof_mortN":  ["M_LIVESTEMN_STORAGE_TO_LITTER",
                              "M_DEADSTEMN_STORAGE_TO_LITTER",
                              "M_LIVESTEMN_XFER_TO_LITTER",
                              "M_DEADSTEMN_XFER_TO_LITTER"],
}

## Core processing function

In [22]:
def process_mortality(ds, include_profiles=False):
    """
    Extract and aggregate mortality variables from a CLM h1 xarray Dataset.

    Parameters
    ----------
    ds : xr.Dataset
        Opened CLM h1 dataset for one year.
    include_profiles : bool
        If True, also extract CROOT_PROF and STEM_PROF (needed for the first
        year of each simulation segment).

    Returns
    -------
    xr.Dataset
    """
    out = xr.Dataset()
    out["mcdate"] = ds["mcdate"]

    if include_profiles:
        out["CROOT_PROF"] = ds["CROOT_PROF"]
        out["STEM_PROF"]  = ds["STEM_PROF"]

    # Variables split between metabolic and structural
    out["split_leaf_prof_mortC"]  = ds["M_LEAFC_TO_LITTER"]
    out["split_froot_prof_mortC"] = ds["M_FROOTC_TO_LITTER"]
    out["split_leaf_prof_mortN"]  = ds["M_LEAFN_TO_LITTER"]
    out["split_froot_prof_mortN"] = ds["M_FROOTN_TO_LITTER"]

    # Summed metabolic C fluxes
    for out_var, src_vars in MET_C.items():
        out[out_var] = sum(ds[v] for v in src_vars)

    # Summed metabolic N fluxes
    for out_var, src_vars in MET_N.items():
        out[out_var] = sum(ds[v] for v in src_vars)

    return out

## Helper: find h1 file for a given year

In [23]:
def find_h1_file(year):
    """
    Find the preprocessed h1 file for a given year.
    Matches any file ending in .clm2.h1.{year}.nc in H1_DIR.
    Raises FileNotFoundError if none or more than one match found.
    """
    pattern = os.path.join(H1_DIR, f"*.clm2.h1.{year}.nc")
    matches = [f for f in glob.glob(pattern) if "spinup" not in f]
    if len(matches) == 0:
        raise FileNotFoundError(f"No h1 file for year {year} in {H1_DIR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple h1 files for year {year}: {matches}")
    return matches[0]

## Process all years

In [24]:
skipped  = []
missing  = []
processed = []

for year in range(YEAR_START, YEAR_END + 1):
    out_path = os.path.join(MORT_DIR, f"mort_{SITE}_{year}.nc")

    if os.path.exists(out_path):
        skipped.append(year)
        continue

    try:
        h1_file = find_h1_file(year)
    except FileNotFoundError:
        missing.append(year)
        continue

    with xr.open_dataset(h1_file) as ds:
        include_profiles = (year == PROFILE_YEAR)
        ds_out = process_mortality(ds, include_profiles=include_profiles)

    ds_out.to_netcdf(out_path)
    processed.append(year)

print(f"Processed : {len(processed)} years")
print(f"Skipped   : {len(skipped)} years (already existed)")
if missing:
    print(f"Missing   : {len(missing)} years — {missing[:10]}{'...' if len(missing)>10 else ''}")

KeyError: "No variable named 'M_DEADCROOTN_STORAGE_TO_LITTER'. Did you mean one of ('M_LIVECROOTN_STORAGE_TO_LITTER', 'M_FROOTN_STORAGE_TO_LITTER', 'M_DEADSTEMN_STORAGE_TO_LITTER', 'M_LIVECROOTC_STORAGE_TO_LITTER', 'M_FROOTC_STORAGE_TO_LITTER', 'M_DEADSTEMC_STORAGE_TO_LITTER', 'M_DEADCROOTN_XFER_TO_LITTER', 'M_LEAFN_STORAGE_TO_LITTER', 'M_LEAFC_STORAGE_TO_LITTER', 'M_DEADCROOTC_XFER_TO_LITTER')?"

## Create spinup mortality file

In [ ]:
spinup_out = os.path.join(MORT_DIR, f"mort_{SITE}_for_spinup.{SPINUP_START}-{SPINUP_END}.nc")

if os.path.exists(spinup_out):
    print(f"Spinup file already exists: {spinup_out}")
else:
    spinup_datasets = []
    for year in range(SPINUP_START, SPINUP_END + 1):
        mort_file = os.path.join(MORT_DIR, f"mort_{SITE}_{year}.nc")
        if not os.path.exists(mort_file):
            print(f"  WARNING: missing spinup year {year}, skipping.")
            continue
        spinup_datasets.append(xr.open_dataset(mort_file))

    if spinup_datasets:
        spinup = xr.concat(spinup_datasets, dim="time")
        spinup.to_netcdf(spinup_out)
        for ds in spinup_datasets:
            ds.close()
        print(f"Spinup file written: {spinup_out}")
    else:
        print("ERROR: No spinup datasets found. Run the processing cells first.")

## Quick sanity check

In [ ]:
# Open a sample year and print variables + time range
sample_year = YEAR_START
sample_file = os.path.join(MORT_DIR, f"mort_{SITE}_{sample_year}.nc")

if os.path.exists(sample_file):
    ds_check = xr.open_dataset(sample_file)
    print(f"Variables in mort_{SITE}_{sample_year}.nc:")
    for v in ds_check.data_vars:
        print(f"  {v:35s}  shape={ds_check[v].shape}")
    ds_check.close()
else:
    print(f"Sample file not found: {sample_file}")